## Checking `robots.txt` before scraping

Since this is a real commercial site (not a scraping sandbox), I checked
`https://www.ruparupa.com/robots.txt` before writing any code.

The target page (`/ms/promo-9-9`) is not listed under any `Disallow` rule for
`User-agent: *`, so scraping it is permitted. The disallowed paths are mostly
private/transactional pages (`/my-account`, `/checkout`, `/cart`, `/dashboard`)
— none of which apply here.

This scrape was a one-time, small-scale run for learning purposes, with a
delay between requests (`time.sleep()`), and the data is not redistributed
or used commercially. 

"This notebook is designed to run in Google Colab (uses Colab-specific commands for installing Chrome/Selenium)."

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

## Setting up Selenium

In [ ]:
!apt-get update
!apt-get install -y chromium-chromedriver
!pip install selenium

In [ ]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y ./google-chrome-stable_current_amd64.deb

In [6]:
!google-chrome --version

Google Chrome 153.0.8010.36 


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--headless")        # no visible browser window 
options.add_argument("--no-sandbox")       # required in Colab's restricted environment
options.add_argument("--disable-dev-shm-usage")  # avoids a common Colab memory crash
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

In [8]:
import time

driver.get("https://www.ruparupa.com/ms/promo-9-9")
print(driver.title)

for i in range(5):
  driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
  time.sleep(2)
  print(f"Scrolled {i+1} times")

Promo 9.9 2026: Diskon & Voucher Belanja | ruparupa
Scrolled 1 times
Scrolled 2 times
Scrolled 3 times
Scrolled 4 times
Scrolled 5 times


## finding how many products there are in product card actually

In [ ]:
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

products = soup.find_all("div", class_="row product-card")
print(len(products))

72


## Checking if there are duplicate rows / objects

In [ ]:
product_names = []

for p in products:
    name_tag = p.find("span", class_="product__name")
    if name_tag:
        product_names.append(name_tag.get_text(strip=True))

print("Total found:", len(product_names))
print("Unique names:", len(set(product_names)))

Total found: 72
Unique names: 71


## remove the duplicated objects

In [ ]:
seen = set()
unique_products = []

for p in products:
    name_tag = p.find("span", class_="product__name")
    if name_tag:
        name = name_tag.get_text(strip=True)
        if name not in seen:
            seen.add(name)
            unique_products.append(p)

print(len(unique_products))

71


## Extracting product data

In [ ]:
def extract_ruparupa_product(product_div):
  ##Product name
  name_tag = product_div.find("span", class_="product__name")
  name = name_tag.get_text(strip=True) if name_tag else None ##strip used for removing space and white lines

  ##Initial Price
  initial_price_tag = product_div.find("div", class_="price__initial")
  initial_price = initial_price_tag.get_text(strip=True) if initial_price_tag else None

  ##Discount
  discount_tag = product_div.find("div", class_="price__discount")
  discount = discount_tag.get_text(strip=True) if discount_tag else None

  ##Product URL
  product_url_tag = product_div.find("a")
  product_url = product_url_tag["href"] if product_url_tag else None

  ##Image Url
  image_url_tag = product_div.find("img")
  image_url = image_url_tag["src"] if image_url_tag else None

  ##Final Price
  final_price_tag = product_div.find("div", class_="price__real")
  final_price = final_price_tag.get_text(strip=True) if final_price_tag else None

  ##Rating, used attrs when you're not using class_ / spans data
  rating_tag = product_div.find("span", attrs={"data-testid":"lblTotalRating"})
  rating = rating_tag.get_text(strip=True) if rating_tag else None

  ##Review
  review_tag = product_div.find("span", attrs={"data-testid":"lblTotalUlasan"})
  review = review_tag.get_text(strip=True) if review_tag else None

  return {"name" : name, "initial_price" : initial_price, "product_url" : product_url,
          "image_url" : image_url,"discount" : discount, "final_price" : final_price,
          "rating" : rating,"review" : review}


In [ ]:
products_data = []

for p in unique_products:
    data = extract_ruparupa_product(p)
    products_data.append(data)

for item in products_data:
  print(item)

## Exporting to CSV

In [17]:
df = pd.DataFrame(products_data)
df.head()

df.to_csv("ruparupa_products.csv", index=False)

In [18]:
from google.colab import files
files.download("ruparupa_products.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
driver.quit()

<bound method LocalWebDriver.quit of <selenium.webdriver.chrome.webdriver.WebDriver (session="b009acd424279da6f702f4d68e03adda")>>